# Publication Readiness Audit

This notebook turns the existing external-lab RefMet evaluation outputs into publication-ready audit artifacts. It does **not** re-embed names, rebuild FAISS, or fine-tune any model. It only reads the generated Part 2 result files and writes clean summaries, review sheets, examples, and plots.

Main outputs are written to:

`results/publication_readiness/`

Run this notebook from top to bottom in Colab after mounting Google Drive.


In [ ]:
# Colab normally includes pandas, numpy, and matplotlib.
# If a custom runtime is missing them, uncomment the next line.
# !pip -q install pandas numpy matplotlib

# Colab setup: mount Drive when running in Google Colab.
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)


In [ ]:
# Imports and path configuration.
import json
import math
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

RANDOM_SEED = 42

# Update PROJECT_DIR manually if your Drive path is different.
CANDIDATE_PROJECT_DIRS = [
    Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
    Path('/content/drive/MyDrive/ChemSynonymizer'),
    Path('/content/chem_synonymizer'),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_DIR = next((p for p in CANDIDATE_PROJECT_DIRS if (p / 'results').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find project directory. Set PROJECT_DIR manually in this cell.')

RUN_DIR = PROJECT_DIR / 'results' / 'external_lab_refmet_model_comparison_reproducible'
OUT_DIR = PROJECT_DIR / 'results' / 'publication_readiness'
PLOTS_DIR = OUT_DIR / 'plots'

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('RUN_DIR:', RUN_DIR)
print('OUT_DIR:', OUT_DIR)

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Canonical Part 2 result folder not found: {RUN_DIR}')


In [ ]:
# Artifact paths used by this audit.
PATHS = {
    'config': RUN_DIR / 'config.json',
    'data_summary': RUN_DIR / 'v4_data_summary.json',
    'final_metrics': RUN_DIR / 'v4_final_metrics.json',
    'final_comparison': RUN_DIR / 'v4_final_comparison.csv',
    'predictions': RUN_DIR / 'v4_final_test_predictions.csv',
    'top_hits': RUN_DIR / 'v4_final_test_top_hits.csv',
    'confidence_curves': RUN_DIR / 'v4_confidence_curves.csv',
    'refmet_cid_mapping_summary': RUN_DIR / 'v4_refmet_cid_mapping_summary.json',
    'deterministic_mapping_summary': RUN_DIR / 'deterministic_cid_mapping' / 'deterministic_mapping_summary.json',
    'refmet_dictionary': RUN_DIR / 'artifacts' / 'v4_refmet_label_dictionary.csv',
    'refmet_cid_mapping': RUN_DIR / 'artifacts' / 'v4_refmet_label_cid_mapping.csv',
    'query_pairs': RUN_DIR / 'artifacts' / 'v4_query_pairs.csv',
    'taxonomy': RUN_DIR / 'step6_error_taxonomy.csv',
    'taxonomy_counts': RUN_DIR / 'step6_error_taxonomy_counts.csv',
    'case_group_counts': RUN_DIR / 'step6_case_group_counts.csv',
    'representative_examples': RUN_DIR / 'step6_representative_examples.csv',
    'reranker_ablation_metrics': RUN_DIR / 'reranker_ablation_metrics.csv',
    'bootstrap_confidence_intervals': PROJECT_DIR / 'results' / 'bootstrap_confidence_intervals.csv',
    'bootstrap_metric_differences': PROJECT_DIR / 'results' / 'bootstrap_metric_differences.csv',
}

missing_required = [name for name in ['data_summary', 'final_metrics', 'predictions'] if not PATHS[name].exists()]
if missing_required:
    raise FileNotFoundError(f'Missing required artifacts: {missing_required}')

print('Artifact availability:')
for name, path in PATHS.items():
    print(f'  {name:32s}', 'OK' if path.exists() else 'missing', path)


In [ ]:
# Helper functions.
def read_json(path: Path, default=None):
    if not path.exists():
        return default if default is not None else {}
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False, **kwargs)


def to_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.fillna(False)
    return s.astype('string').str.lower().isin(['true', '1', 'yes'])


def rank_metrics(df: pd.DataFrame, rank_col: str, ks=(1, 5, 10, 25, 50, 100, 256)) -> dict:
    ranks = pd.to_numeric(df[rank_col], errors='coerce')
    reciprocal = np.where(ranks.notna() & (ranks > 0), 1.0 / ranks, 0.0)
    out = {'MRR': float(np.mean(reciprocal)), 'n': int(len(df))}
    for k in ks:
        out[f'Hit@{k}'] = float(np.mean(ranks.notna() & (ranks <= k)))
    return out


def sample_rows(df: pd.DataFrame, n: int, random_state: int = RANDOM_SEED) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    return df.sample(n=min(n, len(df)), random_state=random_state).copy()


def top_rows(df: pd.DataFrame, n: int, sort_col: str = 'confidence_margin', ascending: bool = False) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    if sort_col in df.columns:
        tmp = df.copy()
        tmp[sort_col] = pd.to_numeric(tmp[sort_col], errors='coerce')
        return tmp.sort_values(sort_col, ascending=ascending).head(n).copy()
    return df.head(n).copy()


def write_json(path: Path, obj) -> None:
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)


def save_bar(df: pd.DataFrame, x: str, y: str, title: str, out_path: Path, rotation: int = 35) -> None:
    plt.figure(figsize=(10, 5))
    plt.bar(df[x].astype(str), df[y])
    plt.title(title)
    plt.xticks(rotation=rotation, ha='right')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


In [ ]:
# Load canonical outputs. These files are small enough for Colab RAM.
config = read_json(PATHS['config'])
data_summary = read_json(PATHS['data_summary'])
final_metrics = read_json(PATHS['final_metrics'])
refmet_mapping_summary = read_json(PATHS['refmet_cid_mapping_summary'])
det_mapping_summary = read_json(PATHS['deterministic_mapping_summary'])

pred = read_csv(PATHS['predictions'])
confidence = read_csv(PATHS['confidence_curves'])
final_comparison = read_csv(PATHS['final_comparison'])
taxonomy = read_csv(PATHS['taxonomy']) if PATHS['taxonomy'].exists() else pd.DataFrame()
taxonomy_counts_file = read_csv(PATHS['taxonomy_counts'])
case_group_counts_file = read_csv(PATHS['case_group_counts'])

print('predictions:', pred.shape)
print('taxonomy:', taxonomy.shape)
print('confidence:', confidence.shape)

for col in ['top1_exact_correct', 'top1_identity_correct', 'metabolite_equals_refmet']:
    if col in pred.columns:
        pred[col + '_bool'] = to_bool_series(pred[col])
    if col in taxonomy.columns:
        taxonomy[col + '_bool'] = to_bool_series(taxonomy[col])

if taxonomy.empty:
    taxonomy = pred.copy()
    taxonomy['primary_error_category'] = np.where(
        taxonomy['top1_exact_correct_bool'],
        'exact_top1_success',
        np.where(taxonomy['top1_identity_correct_bool'], 'identity_match_not_exact_label', 'unclassified_failure')
    )
    taxonomy['case_group'] = taxonomy['primary_error_category']

for rank_col in ['embedding_exact_rank', 'embedding_identity_rank', 'reranker_exact_rank', 'reranker_identity_rank']:
    if rank_col in pred.columns:
        pred[rank_col] = pd.to_numeric(pred[rank_col], errors='coerce')
    if rank_col in taxonomy.columns:
        taxonomy[rank_col] = pd.to_numeric(taxonomy[rank_col], errors='coerce')

if 'confidence_margin' in pred.columns:
    pred['confidence_margin'] = pd.to_numeric(pred['confidence_margin'], errors='coerce')
if 'confidence_margin' in taxonomy.columns:
    taxonomy['confidence_margin'] = pd.to_numeric(taxonomy['confidence_margin'], errors='coerce')


## 1. Dataset and Artifact Manifest

This creates frozen counts and source paths for the paper methods/results section.


In [ ]:
manifest_records = []

def add_manifest(field, value, source, note=''):
    manifest_records.append({'field': field, 'value': value, 'source': str(source), 'note': note})


for field in ['raw_lab_rows', 'rows_with_both_names', 'deduplicated_query_pairs', 'unique_refmet_labels', 'identical_metabolite_refmet_pairs']:
    add_manifest(field, data_summary.get(field), PATHS['data_summary'])

split_counts = data_summary.get('split_counts', {})
for split, count in split_counts.items():
    add_manifest(f'split_{split}_rows', count, PATHS['data_summary'])

for field in ['mapped_labels_total', 'unmapped_labels_total', 'unique_refmet_labels']:
    add_manifest(f'refmet_{field}', refmet_mapping_summary.get(field), PATHS['refmet_cid_mapping_summary'])

mapping_summary = det_mapping_summary.get('mapping_summary', {}) if isinstance(det_mapping_summary, dict) else {}
for field in ['matched_rows', 'unique_cid_rows', 'ambiguous_cid_rows', 'unmatched_rows', 'matched_row_pct', 'unique_cid_row_pct']:
    add_manifest(f'deterministic_{field}', mapping_summary.get(field), PATHS['deterministic_mapping_summary'])

artifact_records = []
for name, path in PATHS.items():
    artifact_records.append({
        'artifact': name,
        'path': str(path),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
    })

dataset_manifest = pd.DataFrame(manifest_records)
artifact_manifest = pd.DataFrame(artifact_records)

dataset_manifest.to_csv(OUT_DIR / 'publication_dataset_manifest.csv', index=False)
artifact_manifest.to_csv(OUT_DIR / 'publication_artifact_manifest.csv', index=False)
write_json(OUT_DIR / 'publication_dataset_manifest.json', {
    'generated_at': datetime.now().isoformat(),
    'project_dir': str(PROJECT_DIR),
    'run_dir': str(RUN_DIR),
    'dataset_summary': data_summary,
    'refmet_cid_mapping_summary': refmet_mapping_summary,
    'deterministic_mapping_summary': det_mapping_summary,
    'artifact_manifest': artifact_records,
})

display(dataset_manifest)


## 2. Primary Metrics and Non-Identical Subset Summary

This recomputes the key metrics directly from per-query ranks so the paper has a simple table tied to the held-out test predictions.


In [ ]:
metric_rows = []
nonidentical_pred = pred[~pred['metabolite_equals_refmet_bool']] if 'metabolite_equals_refmet_bool' in pred.columns else pred.copy()
groups = {
    'all_test_queries': pred,
    'nonidentical_only': nonidentical_pred,
}

for group_name, group_df in groups.items():
    for system_name, rank_col in [
        ('embedding_exact', 'embedding_exact_rank'),
        ('embedding_identity', 'embedding_identity_rank'),
        ('reranker_exact', 'reranker_exact_rank'),
        ('reranker_identity', 'reranker_identity_rank'),
    ]:
        if rank_col in group_df.columns:
            metrics = rank_metrics(group_df, rank_col)
            metric_rows.append({'group': group_name, 'system': system_name, **metrics})

primary_metrics = pd.DataFrame(metric_rows)
primary_metrics.to_csv(OUT_DIR / 'primary_metric_summary.csv', index=False)

plot_df = primary_metrics[(primary_metrics['group'] == 'all_test_queries') & (primary_metrics['system'].isin(['embedding_exact', 'reranker_exact']))]
plot_long = plot_df.melt(id_vars=['system'], value_vars=['MRR', 'Hit@1', 'Hit@10', 'Hit@256'], var_name='metric', value_name='value')
plt.figure(figsize=(9, 5))
for system, sub in plot_long.groupby('system'):
    plt.plot(sub['metric'], sub['value'], marker='o', label=system)
plt.ylim(0, 1)
plt.title('Primary held-out test metrics')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'primary_metrics_comparison.png', dpi=200)
plt.close()

display(primary_metrics)


## 3. Strict Non-Self Evaluation

This stricter audit removes cases where the gold RefMet label is the same normalized surface as the query and also removes query-identical candidates from the retrieved candidate pool before recomputing ranks. This is the fairest version for claiming synonym/variant normalization rather than exact-copy normalization.


In [ ]:
strict_outputs = {}
top_hits = read_csv(PATHS['top_hits'])

required_top_hit_cols = {'query_id', 'rank', 'candidate_refmet_normalized', 'exact_correct', 'identity_correct'}
if top_hits.empty or not required_top_hit_cols.issubset(set(top_hits.columns)):
    print('Top-hit file missing or incomplete; strict no-self-surface ranking skipped.')
else:
    strict_base = pred[pred['metabolite_normalized'].astype('string') != pred['refmet_normalized'].astype('string')].copy()
    strict_query_ids = set(strict_base['query_id'])

    self_summary = {
        'test_queries_total': int(len(pred)),
        'gold_refmet_equals_query_surface': int((pred['metabolite_normalized'].astype('string') == pred['refmet_normalized'].astype('string')).sum()),
        'top1_prediction_equals_query_surface': int((pred['metabolite_normalized'].astype('string') == pred['top1_refmet_normalized'].astype('string')).sum()),
        'nonidentical_gold_but_top1_equals_query_surface': int(((pred['metabolite_normalized'].astype('string') != pred['refmet_normalized'].astype('string')) & (pred['metabolite_normalized'].astype('string') == pred['top1_refmet_normalized'].astype('string'))).sum()),
        'strict_nonidentical_queries': int(len(strict_base)),
    }

    th = top_hits[top_hits['query_id'].isin(strict_query_ids)].copy()
    th['rank'] = pd.to_numeric(th['rank'], errors='coerce')
    th = th.sort_values(['query_id', 'rank']).reset_index(drop=True)
    th = th.merge(strict_base[['query_id', 'metabolite_normalized']], on='query_id', how='left')
    th['candidate_equals_query_surface'] = th['candidate_refmet_normalized'].astype('string') == th['metabolite_normalized'].astype('string')
    th['exact_correct_bool'] = to_bool_series(th['exact_correct'])
    th['identity_correct_bool'] = to_bool_series(th['identity_correct'])

    query_surface_in_pool = th.groupby('query_id')['candidate_equals_query_surface'].any()
    self_summary['strict_queries_with_query_surface_in_top256'] = int(query_surface_in_pool.sum())
    self_summary['strict_query_surface_in_top256_rate'] = float(query_surface_in_pool.mean()) if len(query_surface_in_pool) else 0.0

    strict_hits = th[~th['candidate_equals_query_surface']].copy()
    strict_hits['strict_rank_after_query_surface_removal'] = strict_hits.groupby('query_id').cumcount() + 1

    strict_top1 = strict_hits[strict_hits['strict_rank_after_query_surface_removal'] == 1].copy()
    exact_rank = strict_hits[strict_hits['exact_correct_bool']].groupby('query_id')['strict_rank_after_query_surface_removal'].min()
    identity_rank = strict_hits[strict_hits['identity_correct_bool']].groupby('query_id')['strict_rank_after_query_surface_removal'].min()

    strict_eval = strict_base[['query_id', 'metabolite_name', 'refmet_name', 'metabolite_normalized', 'refmet_normalized']].copy()
    strict_eval = strict_eval.merge(exact_rank.rename('strict_exact_rank'), on='query_id', how='left')
    strict_eval = strict_eval.merge(identity_rank.rename('strict_identity_rank'), on='query_id', how='left')
    strict_eval = strict_eval.merge(
        strict_top1[['query_id', 'target_id', 'candidate_refmet_name', 'candidate_refmet_normalized', 'candidate_refmet_cids', 'exact_correct_bool', 'identity_correct_bool', 'identity_reason', 'embedding_score', 'rerank_score']],
        on='query_id', how='left'
    )
    strict_eval = strict_eval.rename(columns={
        'target_id': 'strict_top1_target_id',
        'candidate_refmet_name': 'strict_top1_refmet_name',
        'candidate_refmet_normalized': 'strict_top1_refmet_normalized',
        'candidate_refmet_cids': 'strict_top1_refmet_cids',
        'exact_correct_bool': 'strict_top1_exact_correct',
        'identity_correct_bool': 'strict_top1_identity_correct',
        'identity_reason': 'strict_top1_identity_reason',
        'embedding_score': 'strict_top1_embedding_score',
        'rerank_score': 'strict_top1_rerank_score',
    })

    strict_metric_rows = []
    for system, col in [('strict_exact_no_self_surface', 'strict_exact_rank'), ('strict_identity_no_self_surface', 'strict_identity_rank')]:
        metrics = rank_metrics(strict_eval, col)
        strict_metric_rows.append({'group': 'nonidentical_gold_and_no_query_surface_candidates', 'system': system, **metrics})
    strict_metrics = pd.DataFrame(strict_metric_rows)

    pd.DataFrame([self_summary]).to_csv(OUT_DIR / 'self_match_leakage_summary.csv', index=False)
    strict_metrics.to_csv(OUT_DIR / 'strict_no_self_surface_metrics.csv', index=False)
    strict_eval.to_csv(OUT_DIR / 'strict_no_self_surface_predictions.csv', index=False)

    strict_outputs = {
        'self_summary': self_summary,
        'strict_metrics': strict_metrics,
        'strict_eval': strict_eval,
    }

    print('Self/surface leakage summary')
    display(pd.DataFrame([self_summary]))
    print('Strict metrics after removing gold self-matches and query-identical candidates')
    display(strict_metrics)

    plot_df = strict_metrics.melt(id_vars=['system'], value_vars=['MRR', 'Hit@1', 'Hit@10', 'Hit@256'], var_name='metric', value_name='value')
    plt.figure(figsize=(8, 5))
    for system, sub in plot_df.groupby('system'):
        plt.plot(sub['metric'], sub['value'], marker='o', label=system)
    plt.ylim(0, 1)
    plt.title('Strict non-self evaluation')
    plt.ylabel('Score')
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'strict_no_self_surface_metrics.png', dpi=200)
    plt.close()


## 4. Conservative Row-Exclusion Evaluation

This audit removes rows that could make the task look easier than it is: cases where the gold `refmet_name` is the same normalized surface as the `metabolite_name`, and cases where the original pipeline's top-1 prediction is the same normalized surface as the query. The resulting subset is a conservative estimate of synonym/variant normalization performance rather than exact-copy lookup performance.


In [ ]:
conservative_outputs = {}
required_cols = {'query_id', 'metabolite_normalized', 'refmet_normalized', 'top1_refmet_normalized'}
if not required_cols.issubset(set(pred.columns)):
    print(f'Conservative row-exclusion audit skipped. Missing columns: {sorted(required_cols - set(pred.columns))}')
else:
    audit_pred = pred.copy()
    query_norm = audit_pred['metabolite_normalized'].astype('string').fillna('')
    gold_norm = audit_pred['refmet_normalized'].astype('string').fillna('')
    top1_norm = audit_pred['top1_refmet_normalized'].astype('string').fillna('')

    gold_self_mask = query_norm.eq(gold_norm)
    original_top1_query_surface_mask = query_norm.eq(top1_norm)
    nonidentical_gold_top1_query_surface_mask = (~gold_self_mask) & original_top1_query_surface_mask
    conservative_exclude_mask = gold_self_mask | original_top1_query_surface_mask

    audit_pred['gold_refmet_equals_query_surface'] = gold_self_mask
    audit_pred['original_top1_equals_query_surface'] = original_top1_query_surface_mask
    audit_pred['nonidentical_gold_but_original_top1_equals_query_surface'] = nonidentical_gold_top1_query_surface_mask
    audit_pred['excluded_from_conservative_no_query_surface_eval'] = conservative_exclude_mask
    audit_pred['conservative_exclusion_reason'] = np.select(
        [gold_self_mask, nonidentical_gold_top1_query_surface_mask],
        ['gold_refmet_equals_query_surface', 'nonidentical_gold_but_original_top1_equals_query_surface'],
        default='kept'
    )

    query_compact = query_norm.str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)
    top1_compact = top1_norm.str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)
    audit_pred['original_top1_compact_equals_query_surface'] = query_compact.eq(top1_compact) & query_compact.ne('')

    conservative_kept = audit_pred.loc[~conservative_exclude_mask].copy()
    conservative_excluded = audit_pred.loc[conservative_exclude_mask].copy()

    if 'top1_exact_correct_bool' not in conservative_kept.columns and 'top1_exact_correct' in conservative_kept.columns:
        conservative_kept['top1_exact_correct_bool'] = to_bool_series(conservative_kept['top1_exact_correct'])
    if 'top1_identity_correct_bool' not in conservative_kept.columns and 'top1_identity_correct' in conservative_kept.columns:
        conservative_kept['top1_identity_correct_bool'] = to_bool_series(conservative_kept['top1_identity_correct'])

    exclusion_summary = {
        'test_queries_total': int(len(audit_pred)),
        'removed_gold_refmet_equals_query_surface': int(gold_self_mask.sum()),
        'removed_original_top1_equals_query_surface': int(original_top1_query_surface_mask.sum()),
        'removed_nonidentical_gold_but_original_top1_equals_query_surface': int(nonidentical_gold_top1_query_surface_mask.sum()),
        'removed_union_gold_self_or_original_top1_query_surface': int(conservative_exclude_mask.sum()),
        'kept_conservative_queries': int(len(conservative_kept)),
        'kept_conservative_fraction': float(len(conservative_kept) / len(audit_pred)) if len(audit_pred) else 0.0,
    }

    metric_rows = []
    for system_name, rank_col in [
        ('embedding_exact', 'embedding_exact_rank'),
        ('embedding_identity', 'embedding_identity_rank'),
        ('reranker_exact', 'reranker_exact_rank'),
        ('reranker_identity', 'reranker_identity_rank'),
    ]:
        if rank_col in conservative_kept.columns:
            metrics = rank_metrics(conservative_kept, rank_col)
            metric_rows.append({
                'group': 'conservative_no_gold_self_or_original_top1_query_surface',
                'system': system_name,
                **metrics,
            })
    conservative_metrics = pd.DataFrame(metric_rows)

    failure_examples = conservative_kept.copy()
    if 'top1_exact_correct_bool' in failure_examples.columns:
        failure_examples = failure_examples[~failure_examples['top1_exact_correct_bool']]
    if 'confidence_margin' in failure_examples.columns:
        failure_examples = failure_examples.sort_values('confidence_margin', ascending=False)

    near_copy_failures = conservative_kept.copy()
    if 'top1_exact_correct_bool' in near_copy_failures.columns:
        near_copy_failures = near_copy_failures[~near_copy_failures['top1_exact_correct_bool']]
    near_copy_failures = near_copy_failures[near_copy_failures['original_top1_compact_equals_query_surface']].copy()
    if 'confidence_margin' in near_copy_failures.columns:
        near_copy_failures = near_copy_failures.sort_values('confidence_margin', ascending=False)

    selected_review_cols = [
        'query_id', 'metabolite_name', 'refmet_name', 'top1_refmet_name',
        'metabolite_normalized', 'refmet_normalized', 'top1_refmet_normalized',
        'embedding_exact_rank', 'reranker_exact_rank', 'reranker_identity_rank',
        'top1_exact_correct', 'top1_identity_correct', 'top1_identity_reason',
        'confidence_margin', 'conservative_exclusion_reason',
        'original_top1_compact_equals_query_surface',
    ]
    selected_review_cols = [c for c in selected_review_cols if c in audit_pred.columns]

    pd.DataFrame([exclusion_summary]).to_csv(OUT_DIR / 'conservative_row_exclusion_summary.csv', index=False)
    conservative_metrics.to_csv(OUT_DIR / 'conservative_no_query_surface_metrics.csv', index=False)
    conservative_kept.to_csv(OUT_DIR / 'conservative_no_query_surface_predictions.csv', index=False)
    conservative_excluded.to_csv(OUT_DIR / 'excluded_self_or_query_surface_rows.csv', index=False)
    failure_examples[selected_review_cols].head(500).to_csv(OUT_DIR / 'conservative_no_query_surface_failure_examples.csv', index=False)
    near_copy_failures[selected_review_cols].head(500).to_csv(OUT_DIR / 'near_query_surface_competitor_examples.csv', index=False)

    if not conservative_metrics.empty:
        plot_df = conservative_metrics[conservative_metrics['system'].isin(['embedding_exact', 'reranker_exact', 'reranker_identity'])]
        plot_long = plot_df.melt(id_vars=['system'], value_vars=['MRR', 'Hit@1', 'Hit@10', 'Hit@256'], var_name='metric', value_name='value')
        plt.figure(figsize=(9, 5))
        for system, sub in plot_long.groupby('system'):
            plt.plot(sub['metric'], sub['value'], marker='o', label=system)
        plt.ylim(0, 1)
        plt.title('Conservative evaluation: no gold self-match or original top-1 query surface')
        plt.ylabel('Score')
        plt.legend()
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / 'conservative_no_query_surface_metrics.png', dpi=200)
        plt.close()

    conservative_outputs = {
        'exclusion_summary': exclusion_summary,
        'metrics': conservative_metrics,
        'kept': conservative_kept,
        'excluded': conservative_excluded,
        'near_copy_failures': near_copy_failures,
    }

    print('Conservative row-exclusion summary')
    display(pd.DataFrame([exclusion_summary]))
    print('Conservative metrics after dropping gold self-matches and original top-1 query-surface rows')
    display(conservative_metrics)
    print('Near-copy top-1 failures remaining after exact row exclusion:', len(near_copy_failures))


## 5. Confidence Threshold Table

This creates the table needed to support a selective prediction workflow: accept high-confidence predictions automatically and send low-confidence predictions to review.


In [ ]:
confidence_out = confidence.copy()
if not confidence_out.empty:
    confidence_out['top1_accuracy_percent'] = 100 * pd.to_numeric(confidence_out['top1_accuracy_when_answered'], errors='coerce')
    confidence_out['coverage_percent'] = 100 * pd.to_numeric(confidence_out['coverage'], errors='coerce')
    confidence_out.to_csv(OUT_DIR / 'confidence_threshold_table.csv', index=False)

    pivot = confidence_out.pivot_table(index='coverage_percent', columns='correctness_definition', values='top1_accuracy_percent').reset_index()
    pivot = pivot.sort_values('coverage_percent', ascending=False)
    pivot.to_csv(OUT_DIR / 'confidence_threshold_table_pivot.csv', index=False)

    plt.figure(figsize=(8, 5))
    for correctness, sub in confidence_out.groupby('correctness_definition'):
        sub = sub.sort_values('coverage')
        plt.plot(sub['coverage_percent'], sub['top1_accuracy_percent'], marker='o', label=correctness)
    plt.gca().invert_xaxis()
    plt.xlabel('Coverage answered (%)')
    plt.ylabel('Top-1 accuracy when answered (%)')
    plt.title('Confidence threshold curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'confidence_threshold_curve.png', dpi=200)
    plt.close()
    display(pivot)
else:
    print('No confidence curve file found.')


## 6. RefMet-to-CID Mapping Audit

This summarizes mapped, unmapped, and ambiguous RefMet labels. The sampled CSVs are useful for a targeted manual check.


In [ ]:
refmet_dict = read_csv(PATHS['refmet_dictionary'])
mapping_audit_summary = pd.DataFrame()

if not refmet_dict.empty:
    refmet_dict['refmet_mapped_to_cid_bool'] = to_bool_series(refmet_dict['refmet_mapped_to_cid']) if 'refmet_mapped_to_cid' in refmet_dict.columns else False
    if 'refmet_n_candidate_cids' in refmet_dict.columns:
        refmet_dict['refmet_n_candidate_cids'] = pd.to_numeric(refmet_dict['refmet_n_candidate_cids'], errors='coerce').fillna(0).astype(int)
    else:
        refmet_dict['refmet_n_candidate_cids'] = 0

    mapping_audit_summary = (
        refmet_dict.groupby(['refmet_mapped_to_cid_bool', 'refmet_mapping_method'], dropna=False)
        .agg(n_labels=('target_id', 'count'), n_query_pairs=('n_query_pairs', 'sum'))
        .reset_index()
        .sort_values(['refmet_mapped_to_cid_bool', 'n_labels'], ascending=[False, False])
    )
    mapping_audit_summary.to_csv(OUT_DIR / 'refmet_cid_mapping_audit.csv', index=False)

    mapped_sample = sample_rows(refmet_dict[refmet_dict['refmet_mapped_to_cid_bool']], 100)
    unmapped_sample = sample_rows(refmet_dict[~refmet_dict['refmet_mapped_to_cid_bool']], 100)
    ambiguous_sample = sample_rows(refmet_dict[refmet_dict['refmet_n_candidate_cids'] > 1], 100)

    mapped_sample.to_csv(OUT_DIR / 'refmet_mapped_sample.csv', index=False)
    unmapped_sample.to_csv(OUT_DIR / 'refmet_unmapped_sample.csv', index=False)
    ambiguous_sample.to_csv(OUT_DIR / 'refmet_ambiguous_mapping_sample.csv', index=False)

    plot_map = refmet_dict.assign(mapping_status=np.where(refmet_dict['refmet_mapped_to_cid_bool'], 'mapped', 'unmapped'))
    counts = plot_map['mapping_status'].value_counts().rename_axis('mapping_status').reset_index(name='n_labels')
    save_bar(counts, 'mapping_status', 'n_labels', 'RefMet target CID mapping coverage', PLOTS_DIR / 'refmet_cid_mapping_coverage.png', rotation=0)
    display(mapping_audit_summary)
else:
    print('RefMet dictionary not found; mapping audit skipped.')


## 7. Error Taxonomy Counts

This recreates count tables from the row-level taxonomy and saves a publication-friendly plot.


In [ ]:
failure_tax = taxonomy[~taxonomy['top1_exact_correct_bool']] if 'top1_exact_correct_bool' in taxonomy.columns else taxonomy

taxonomy_counts = (
    failure_tax.groupby('primary_error_category', dropna=False)
    .agg(
        n_cases=('query_id', 'count'),
        mean_confidence_margin=('confidence_margin', 'mean'),
        identity_correct_rate=('top1_identity_correct_bool', 'mean'),
    )
    .reset_index()
    .sort_values('n_cases', ascending=False)
)
taxonomy_counts['pct_of_exact_failures'] = taxonomy_counts['n_cases'] / max(1, len(failure_tax))
taxonomy_counts.to_csv(OUT_DIR / 'error_taxonomy_summary.csv', index=False)

case_group_counts = (
    taxonomy.groupby('case_group', dropna=False)
    .agg(n_cases=('query_id', 'count'))
    .reset_index()
    .sort_values('n_cases', ascending=False)
)
case_group_counts['pct_of_test'] = case_group_counts['n_cases'] / max(1, len(taxonomy))
case_group_counts.to_csv(OUT_DIR / 'case_group_summary.csv', index=False)

save_bar(taxonomy_counts, 'primary_error_category', 'n_cases', 'Exact top-1 failure taxonomy', PLOTS_DIR / 'error_taxonomy_counts.png', rotation=35)
save_bar(case_group_counts, 'case_group', 'n_cases', 'Final test case groups', PLOTS_DIR / 'case_group_counts.png', rotation=35)

display(taxonomy_counts)


## 8. Stratified Manual Review Sheet

This creates a worksheet for human review. The notebook intentionally leaves manual columns blank. Fill them in later, save as `manual_review_sheet_labeled.csv`, and rerun the final manual-summary cell.


In [ ]:
review_frames = []

def add_review_stratum(name: str, frame: pd.DataFrame, n: int, method: str = 'random'):
    if frame.empty:
        return
    sampled = top_rows(frame, n) if method == 'top_confidence' else sample_rows(frame, n)
    sampled = sampled.copy()
    sampled.insert(0, 'review_stratum', name)
    sampled.insert(1, 'review_sampling_method', method)
    review_frames.append(sampled)


strict_noncopy_mask = (
    (taxonomy['metabolite_normalized'].astype('string') != taxonomy['refmet_normalized'].astype('string'))
    & (taxonomy['metabolite_normalized'].astype('string') != taxonomy['top1_refmet_normalized'].astype('string'))
)
strict_exact_success_pool = taxonomy[taxonomy['top1_exact_correct_bool'] & strict_noncopy_mask]
strict_identity_only_pool = taxonomy[(~taxonomy['top1_exact_correct_bool']) & (taxonomy['top1_identity_correct_bool']) & strict_noncopy_mask]

add_review_stratum('strict_noncopy_exact_top1_success', strict_exact_success_pool, 50)
add_review_stratum('strict_noncopy_identity_aware_only_success', strict_identity_only_pool, 50)
add_review_stratum('gold_in_top10_not_rank1', taxonomy[taxonomy['primary_error_category'].eq('reranking_error_gold_in_top10')], 75)
add_review_stratum('candidate_generation_miss_or_low_rank', taxonomy[taxonomy['primary_error_category'].eq('candidate_generation_miss_or_low_rank')], 75)
add_review_stratum('high_confidence_wrong_identity', taxonomy[(~taxonomy['top1_exact_correct_bool']) & (~taxonomy['top1_identity_correct_bool'])], 50, method='top_confidence')

manual_review = pd.concat(review_frames, ignore_index=True) if review_frames else pd.DataFrame()
manual_review = manual_review.drop_duplicates(subset=['query_id'], keep='first')

review_cols = [
    'review_stratum', 'review_sampling_method', 'query_id', 'source_row_id', 'split',
    'metabolite_name', 'refmet_name', 'top1_refmet_name',
    'metabolite_normalized', 'refmet_normalized', 'top1_refmet_normalized',
    'top1_exact_correct', 'top1_identity_correct', 'top1_identity_reason',
    'primary_error_category', 'case_group', 'taxonomy_evidence',
    'embedding_exact_rank', 'reranker_exact_rank', 'reranker_identity_rank',
    'confidence_margin', 'gold_refmet_cids', 'top1_refmet_cids',
]
review_cols = [c for c in review_cols if c in manual_review.columns]
manual_review = manual_review[review_cols].copy()

for col in [
    'manual_exact_label_correct',
    'manual_same_compound_correct',
    'manual_chemically_acceptable',
    'manual_failure_category',
    'manual_confidence_bucket',
    'manual_notes',
    'reviewer',
]:
    manual_review[col] = ''

manual_review_path = OUT_DIR / 'manual_review_sheet.csv'
manual_review.to_csv(manual_review_path, index=False)

review_counts = manual_review['review_stratum'].value_counts().rename_axis('review_stratum').reset_index(name='n_cases') if not manual_review.empty else pd.DataFrame()
review_counts.to_csv(OUT_DIR / 'manual_review_sheet_composition.csv', index=False)
if not review_counts.empty:
    save_bar(review_counts, 'review_stratum', 'n_cases', 'Manual review sheet composition', PLOTS_DIR / 'manual_review_sheet_composition.png', rotation=30)

print('Manual review sheet:', manual_review_path)
display(review_counts)


## 9. Representative Example Tables

These tables are intended for the paper, slides, and advisor discussion.


In [ ]:
example_frames = []

def add_examples(label: str, frame: pd.DataFrame, n: int = 5, method: str = 'top_confidence'):
    if frame.empty:
        return
    chosen = top_rows(frame, n) if method == 'top_confidence' else sample_rows(frame, n)
    chosen = chosen.copy()
    chosen.insert(0, 'example_group', label)
    example_frames.append(chosen)


strict_noncopy_mask = (
    (taxonomy['metabolite_normalized'].astype('string') != taxonomy['refmet_normalized'].astype('string'))
    & (taxonomy['metabolite_normalized'].astype('string') != taxonomy['top1_refmet_normalized'].astype('string'))
)
add_examples('strict_noncopy_exact_success', taxonomy[(taxonomy['top1_exact_correct_bool']) & strict_noncopy_mask], 5)
add_examples('strict_noncopy_reranker_rescued_exact', taxonomy[(taxonomy['top1_exact_correct_bool']) & strict_noncopy_mask & (taxonomy['embedding_exact_rank'].fillna(999999) > 1)], 5)
add_examples('strict_noncopy_identity_aware_success_not_exact', taxonomy[(~taxonomy['top1_exact_correct_bool']) & (taxonomy['top1_identity_correct_bool']) & strict_noncopy_mask], 5)
add_examples('gold_in_top10_not_rank1', taxonomy[taxonomy['primary_error_category'].eq('reranking_error_gold_in_top10')], 5)
add_examples('candidate_generation_miss_or_low_rank', taxonomy[taxonomy['primary_error_category'].eq('candidate_generation_miss_or_low_rank')], 5)
add_examples('high_confidence_wrong_identity', taxonomy[(~taxonomy['top1_exact_correct_bool']) & (~taxonomy['top1_identity_correct_bool'])], 5)

representative_examples = pd.concat(example_frames, ignore_index=True) if example_frames else pd.DataFrame()
keep_cols = [
    'example_group', 'query_id', 'metabolite_name', 'refmet_name', 'top1_refmet_name',
    'top1_exact_correct', 'top1_identity_correct', 'top1_identity_reason',
    'primary_error_category', 'case_group', 'embedding_exact_rank', 'reranker_exact_rank',
    'reranker_identity_rank', 'confidence_margin', 'gold_refmet_cids', 'top1_refmet_cids',
    'taxonomy_evidence',
]
keep_cols = [c for c in keep_cols if c in representative_examples.columns]
representative_examples = representative_examples[keep_cols].copy()
representative_examples.to_csv(OUT_DIR / 'representative_success_failure_examples.csv', index=False)

reranker_rescues = taxonomy[(taxonomy['top1_exact_correct_bool']) & strict_noncopy_mask & (taxonomy['embedding_exact_rank'].fillna(999999) > 1)].copy()
top_rows(reranker_rescues, 200).to_csv(OUT_DIR / 'reranker_rescue_examples.csv', index=False)

candidate_failures = taxonomy[taxonomy['primary_error_category'].eq('candidate_generation_miss_or_low_rank')].copy()
sample_rows(candidate_failures, 200).to_csv(OUT_DIR / 'candidate_generation_failure_examples.csv', index=False)

gold_in_top10 = taxonomy[taxonomy['primary_error_category'].eq('reranking_error_gold_in_top10')].copy()
top_rows(gold_in_top10, 200).to_csv(OUT_DIR / 'gold_in_top10_not_rank1_examples.csv', index=False)

high_conf_wrong = taxonomy[(~taxonomy['top1_exact_correct_bool']) & (~taxonomy['top1_identity_correct_bool'])].copy()
top_rows(high_conf_wrong, 200).to_csv(OUT_DIR / 'high_confidence_wrong_examples.csv', index=False)

display(representative_examples.head(30))


## 10. Bootstrap and Ablation Tables for Reporting

This copies the most important statistical-support tables into the publication-readiness folder.


In [ ]:
bootstrap_ci = read_csv(PATHS['bootstrap_confidence_intervals'])
bootstrap_diff = read_csv(PATHS['bootstrap_metric_differences'])
ablation = read_csv(PATHS['reranker_ablation_metrics'])

if not bootstrap_ci.empty:
    bootstrap_ci.to_csv(OUT_DIR / 'bootstrap_confidence_intervals.csv', index=False)
if not bootstrap_diff.empty:
    bootstrap_diff.to_csv(OUT_DIR / 'bootstrap_metric_differences.csv', index=False)
if not ablation.empty:
    ablation.to_csv(OUT_DIR / 'reranker_ablation_metrics.csv', index=False)

print('bootstrap_ci:', bootstrap_ci.shape)
print('bootstrap_diff:', bootstrap_diff.shape)
print('ablation:', ablation.shape)

display(ablation.head(12) if not ablation.empty else pd.DataFrame())


## 11. Publication Readiness Summary

This writes a short Markdown summary that can be pasted into project notes or used as a checklist before writing.


In [ ]:
def get_metric(group: str, system: str, metric: str):
    row = primary_metrics[(primary_metrics['group'] == group) & (primary_metrics['system'] == system)]
    if row.empty or metric not in row.columns:
        return None
    return float(row.iloc[0][metric])


def format_metric(value, scale=1.0, suffix=''):
    if value is None or pd.isna(value):
        return 'NA'
    return f'{scale * float(value):.4f}{suffix}'


summary_lines = []
summary_lines.append('# Publication Readiness Audit Summary')
summary_lines.append('')
summary_lines.append(f'Generated: {datetime.now().isoformat()}')
summary_lines.append('')
summary_lines.append('## Dataset')
summary_lines.append(f"- Raw lab rows: {data_summary.get('raw_lab_rows')}")
summary_lines.append(f"- Rows with both names: {data_summary.get('rows_with_both_names')}")
summary_lines.append(f"- Deduplicated query-target pairs: {data_summary.get('deduplicated_query_pairs')}")
summary_lines.append(f"- Unique RefMet target labels: {data_summary.get('unique_refmet_labels')}")
summary_lines.append(f"- Split counts: {data_summary.get('split_counts')}")
summary_lines.append('')
summary_lines.append('## Main Test Results')
summary_lines.append(f"- Embedding-only exact MRR: {format_metric(get_metric('all_test_queries', 'embedding_exact', 'MRR'))}")
summary_lines.append(f"- Embedding-only exact Hit@1: {format_metric(get_metric('all_test_queries', 'embedding_exact', 'Hit@1'), 100, '%')}")
summary_lines.append(f"- Reranker exact MRR: {format_metric(get_metric('all_test_queries', 'reranker_exact', 'MRR'))}")
summary_lines.append(f"- Reranker exact Hit@1: {format_metric(get_metric('all_test_queries', 'reranker_exact', 'Hit@1'), 100, '%')}")
summary_lines.append(f"- Reranker exact Hit@10: {format_metric(get_metric('all_test_queries', 'reranker_exact', 'Hit@10'), 100, '%')}")
summary_lines.append('')
if strict_outputs:
    sm = strict_outputs.get('self_summary', {})
    strict_metrics_df = strict_outputs.get('strict_metrics', pd.DataFrame())
    exact_row = strict_metrics_df[strict_metrics_df['system'].eq('strict_exact_no_self_surface')]
    identity_row = strict_metrics_df[strict_metrics_df['system'].eq('strict_identity_no_self_surface')]
    summary_lines.append('## Strict Non-Self Evaluation')
    summary_lines.append(f"- Gold RefMet equals query surface in test set: {sm.get('gold_refmet_equals_query_surface')}")
    summary_lines.append(f"- Top-1 prediction equals query surface: {sm.get('top1_prediction_equals_query_surface')}")
    summary_lines.append(f"- Strict non-identical test queries: {sm.get('strict_nonidentical_queries')}")
    if not exact_row.empty:
        summary_lines.append(f"- Strict exact MRR after removing query-identical candidates: {float(exact_row.iloc[0]['MRR']):.4f}")
        summary_lines.append(f"- Strict exact Hit@1 after removing query-identical candidates: {100 * float(exact_row.iloc[0]['Hit@1']):.2f}%")
        summary_lines.append(f"- Strict exact Hit@10 after removing query-identical candidates: {100 * float(exact_row.iloc[0]['Hit@10']):.2f}%")
    if not identity_row.empty:
        summary_lines.append(f"- Strict identity-aware Hit@1 after removing query-identical candidates: {100 * float(identity_row.iloc[0]['Hit@1']):.2f}%")
    summary_lines.append('')
if conservative_outputs:
    cs = conservative_outputs.get('exclusion_summary', {})
    conservative_metrics_df = conservative_outputs.get('metrics', pd.DataFrame())
    exact_row = conservative_metrics_df[conservative_metrics_df['system'].eq('reranker_exact')]
    identity_row = conservative_metrics_df[conservative_metrics_df['system'].eq('reranker_identity')]
    summary_lines.append('## Conservative Row-Exclusion Evaluation')
    summary_lines.append(f"- Removed gold self-match rows: {cs.get('removed_gold_refmet_equals_query_surface')}")
    summary_lines.append(f"- Removed rows where original top-1 equals query surface: {cs.get('removed_original_top1_equals_query_surface')}")
    summary_lines.append(f"- Removed non-identical gold rows where original top-1 equals query surface: {cs.get('removed_nonidentical_gold_but_original_top1_equals_query_surface')}")
    summary_lines.append(f"- Kept conservative test queries: {cs.get('kept_conservative_queries')}")
    if not exact_row.empty:
        summary_lines.append(f"- Conservative reranker exact MRR: {float(exact_row.iloc[0]['MRR']):.4f}")
        summary_lines.append(f"- Conservative reranker exact Hit@1: {100 * float(exact_row.iloc[0]['Hit@1']):.2f}%")
        summary_lines.append(f"- Conservative reranker exact Hit@10: {100 * float(exact_row.iloc[0]['Hit@10']):.2f}%")
    if not identity_row.empty:
        summary_lines.append(f"- Conservative reranker identity-aware Hit@1: {100 * float(identity_row.iloc[0]['Hit@1']):.2f}%")
    summary_lines.append('')
summary_lines.append('## Manual Review Sheet')
summary_lines.append(f"- Rows generated for manual review: {len(manual_review)}")
summary_lines.append('- File: `manual_review_sheet.csv`')
summary_lines.append('- Fill manual columns, save as `manual_review_sheet_labeled.csv`, then rerun the final manual-summary cell.')
summary_lines.append('')
summary_lines.append('## Key Output Files')
for file_name in [
    'publication_dataset_manifest.csv',
    'primary_metric_summary.csv',
    'self_match_leakage_summary.csv',
    'strict_no_self_surface_metrics.csv',
    'strict_no_self_surface_predictions.csv',
    'conservative_row_exclusion_summary.csv',
    'conservative_no_query_surface_metrics.csv',
    'conservative_no_query_surface_predictions.csv',
    'excluded_self_or_query_surface_rows.csv',
    'conservative_no_query_surface_failure_examples.csv',
    'near_query_surface_competitor_examples.csv',
    'confidence_threshold_table.csv',
    'refmet_cid_mapping_audit.csv',
    'error_taxonomy_summary.csv',
    'manual_review_sheet.csv',
    'representative_success_failure_examples.csv',
    'reranker_rescue_examples.csv',
    'candidate_generation_failure_examples.csv',
]:
    summary_lines.append(f'- `{file_name}`')
summary_lines.append('')
summary_lines.append('## Remaining Human Work')
summary_lines.append('- Manually label the review sheet for chemical acceptability, same-compound correctness, and failure category.')
summary_lines.append('- Use manually confirmed examples in the paper and advisor update slides.')
summary_lines.append('- Treat unseen-compound generalization as a future experiment unless a lightweight diagnostic can be produced from existing cached outputs.')

summary_md = '\n'.join(summary_lines)
(OUT_DIR / 'publication_readiness_summary.md').write_text(summary_md, encoding='utf-8')
print(summary_md)


## 12. Optional: Summarize Completed Manual Labels

After filling the manual review sheet, save it as `results/publication_readiness/manual_review_sheet_labeled.csv` and rerun this cell.


In [ ]:
labeled_path = OUT_DIR / 'manual_review_sheet_labeled.csv'
if labeled_path.exists():
    labeled = pd.read_csv(labeled_path, low_memory=False)
    summary_tables = []
    for col in ['manual_exact_label_correct', 'manual_same_compound_correct', 'manual_chemically_acceptable', 'manual_failure_category']:
        if col in labeled.columns:
            tab = labeled.groupby(['review_stratum', col], dropna=False).size().reset_index(name='n')
            tab.to_csv(OUT_DIR / f'{col}_summary.csv', index=False)
            summary_tables.append((col, tab))
    for col, tab in summary_tables:
        print('\n', col)
        display(tab)
else:
    print('No labeled manual review sheet found yet.')
    print('Fill this file first:', OUT_DIR / 'manual_review_sheet.csv')
    print('Then save a copy as:', labeled_path)
